# CAISO Load Forecasting — Colab Runner

Runs all forecasting models against any calendar month.  
**GPU runtime recommended** (Runtime → Change runtime type → T4 GPU).

### Setup order
1. Upload `filtered.csv` to Google Drive at `My Drive/load_forecasting/filtered.csv`
2. Run **cell 0** (installs) — takes ~2 min on a fresh runtime
3. Run **cell 1** (clone + mount Drive)
4. Run **cell 2** (config) — sets `PREDICT_MONTH`
5. Run individual model cells in section 3, or the batch cell in section 6

### Papers
| Model | Paper |
|---|---|
| GAM | Fan & Hyndman (2012) — [doi:10.1109/TPWRS.2011.2162082](https://doi.org/10.1109/TPWRS.2011.2162082) |
| TFT | Giacomazzi et al. (2023) — [arXiv:2305.10559](https://arxiv.org/abs/2305.10559) |
| ST-CALNet | Cavus & Allahham (2025) — [doi:10.3390/electronics14132514](https://doi.org/10.3390/electronics14132514) |
| BiLSTM / LSTM / Transformer | Dong et al. (2025) survey — [arXiv:2408.16202](https://arxiv.org/abs/2408.16202) |
| LightGBM, XGBoost, RF | Nti et al. (2020) review — [doi:10.1186/s43067-020-00021-8](https://doi.org/10.1186/s43067-020-00021-8) |

## 0. Install dependencies

In [ ]:
import subprocess, sys

pkgs = ['pygam', 'prophet', 'lightgbm', 'xgboost', 'pytorch-forecasting', 'lightning']
for pkg in pkgs:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f'[WARN] {pkg} install failed:\n{r.stderr[:300]}')
    else:
        print(f'  OK: {pkg}')

print('\nVerifying imports...')
checks = {
    'pygam':   'from pygam import LinearGAM',
    'prophet': 'from prophet import Prophet',
    'lightgbm':'import lightgbm',
    'xgboost': 'import xgboost',
    'torch':   'import torch',
    'pytorch_forecasting': 'import pytorch_forecasting',
}
for name, stmt in checks.items():
    try:
        exec(stmt)
        print(f'  OK: {name}')
    except ImportError as e:
        print(f'  MISSING: {name} — {e}')

## 1. Clone repo and mount Google Drive

In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/Chahnwoo/Load-Forecasting.git'
REPO_DIR = '/content/Load-Forecasting'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'Repo ready. cwd = {os.getcwd()}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Edit this path if you stored filtered.csv elsewhere on Drive ──
DRIVE_CSV = '/content/drive/MyDrive/load_forecasting/filtered.csv'

os.makedirs('data/processed', exist_ok=True)
subprocess.run(['cp', DRIVE_CSV, 'data/processed/filtered.csv'], check=True)
print(f'Data ready: {os.path.getsize("data/processed/filtered.csv") / 1e6:.0f} MB')

## 2. Configuration
Set the month to forecast. All model cells use these variables.

In [ ]:
import os, sys, subprocess

# ── Change this to any YYYY-MM you want to forecast ──
PREDICT_MONTH = '2025-01'

CSV_PATH    = 'data/processed/filtered.csv'
SCRIPT_PATH = 'src/modeling/train_forecaster.py'
OUTPUT_DIR  = 'outputs/model_runs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

PYTHON = sys.executable

def run_model(model_name, extra_args=''):
    pred_out    = f'{OUTPUT_DIR}/{model_name}_{PREDICT_MONTH}_predictions.csv'
    metrics_out = f'{OUTPUT_DIR}/{model_name}_{PREDICT_MONTH}_metrics.csv'
    cmd = (
        f'{PYTHON} {SCRIPT_PATH} {CSV_PATH}'
        f' --model {model_name}'
        f' --predict_month {PREDICT_MONTH}'
        f' --output_predictions {pred_out}'
        f' --output_metrics {metrics_out}'
        + (f' {extra_args}' if extra_args else '')
    )
    print(f'\n{"="*60}')
    print(f'Running {model_name} for {PREDICT_MONTH}')
    print(f'{"="*60}')
    sys.stdout.flush()

    # Stream stdout+stderr line-by-line so output is visible in the notebook
    proc = subprocess.Popen(
        cmd, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()

    if proc.returncode != 0:
        print(f'\n[ERROR] {model_name} exited with code {proc.returncode}')
    else:
        print(f'\n[OK] {model_name} complete.')
    return pred_out, metrics_out

print(f'Config ready. Forecasting: {PREDICT_MONTH}')
print(f'Python: {PYTHON}')
print(f'Script: {os.path.abspath(SCRIPT_PATH)}')
print(f'Data:   {os.path.abspath(CSV_PATH)} ({os.path.getsize(CSV_PATH)/1e6:.0f} MB)')

## 3. Models
Each cell below runs one model independently.  
Predictions and metrics are saved to `outputs/model_runs/`.

### 3a. GAM — Semi-parametric additive model
Fan & Hyndman (2012). Best MAPE in our benchmark (~7.9% avg across 2025).  
Uses spline terms for temperature, radiation, lagged load; cyclic splines for hour and day-of-year.

In [ ]:
gam_pred, gam_metrics = run_model('gam', '--gam_n_splines 25 --gam_lam 0.6')

### 3b. XGBoost
Gradient-boosted trees. Second-best in our benchmark.

In [ ]:
xgb_pred, xgb_metrics = run_model('xgboost')

### 3c. LightGBM
Faster gradient boosting with leaf-wise tree growth.

In [ ]:
lgbm_pred, lgbm_metrics = run_model(
    'lightgbm',
    '--lgbm_n_estimators 300 --lgbm_learning_rate 0.05 --lgbm_num_leaves 31'
)

### 3d. LightGBM + XGBoost Ensemble
Weighted average of both boosters. Reduces variance compared to either alone.

In [ ]:
ens_pred, ens_metrics = run_model(
    'lgbm_xgb',
    '--lgbm_n_estimators 300 --lgbm_learning_rate 0.05 --lgbm_num_leaves 31 '
    '--xgb_n_estimators 300 --xgb_learning_rate 0.05 --ensemble_weight 0.5'
)

### 3e. Random Forest
Bagged trees — good interpretable baseline.

In [ ]:
rf_pred, rf_metrics = run_model('random_forest', '--rf_n_estimators 300')

### 3f. Prophet
Facebook Prophet: trend + yearly/weekly/daily seasonality + weather regressors.  
Fits one model per region. ~5–10 min.

In [ ]:
prophet_pred, prophet_metrics = run_model(
    'prophet',
    '--prophet_changepoint_prior_scale 0.05 --prophet_seasonality_prior_scale 10.0'
)

### 3g. LSTM
Vanilla unidirectional LSTM over a 24-hour lookback window. ~10 min on T4.

In [ ]:
lstm_pred, lstm_metrics = run_model(
    'lstm',
    '--lookback 24 --epochs 10 --batch_size 128 --lr 0.001 '
    '--hidden_dim 64 --num_layers 2 --dropout 0.1'
)

### 3h. BiLSTM — Bidirectional LSTM
Reads the sequence in both directions; head takes `hidden_dim * 2` input.

In [ ]:
bilstm_pred, bilstm_metrics = run_model(
    'bilstm',
    '--lookback 24 --epochs 10 --batch_size 128 --lr 0.001 '
    '--hidden_dim 64 --num_layers 2 --dropout 0.1'
)

### 3i. Transformer
Multi-head self-attention encoder over the lookback window.

In [ ]:
tfm_pred, tfm_metrics = run_model(
    'transformer',
    '--lookback 24 --epochs 10 --batch_size 128 --lr 0.001 '
    '--d_model 64 --nhead 4 --num_layers 2 --dim_feedforward 128 --dropout 0.1'
)

### 3j. ST-CALNet — Spatio-Temporal CNN-Attention-LSTM
Cavus & Allahham (2025): Conv1d → LSTM → scaled-dot-product attention → LayerNorm → LSTM → Dense.  
~30–60 min on T4.

In [ ]:
stcalnet_pred, stcalnet_metrics = run_model(
    'stcalnet',
    '--lookback 24 --epochs 10 --batch_size 128 --lr 0.001 '
    '--hidden_dim 64 --num_layers 2 --cnn_channels 64 --dropout 0.1'
)

### 3k. Temporal Fusion Transformer (TFT)
Giacomazzi et al. (2023). LSTM encoder + variable-selection + multi-head attention.  
Target MAPE: ~2.43%. Requires `pytorch-forecasting`. ~15–30 min on T4.

In [ ]:
import os, sys
import numpy as np
import pandas as pd

try:
    from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
    from pytorch_forecasting.data import GroupNormalizer
    from pytorch_forecasting.metrics import MAPE as TFT_MAPE
    import lightning as L
    _tft_ok = True
except ImportError as e:
    _tft_ok = False
    print(f'[SKIP] pytorch-forecasting not available: {e}')
    print('Run cell 0 first to install all dependencies.')

if _tft_ok:
    from src.modeling.train_forecaster import (
        set_seed, build_features, parse_predict_month
    )
    set_seed(42)

    df_raw = pd.read_csv(CSV_PATH)
    month_start, next_month_start, _ = parse_predict_month(PREDICT_MONTH)
    df_raw['time_utc'] = pd.to_datetime(df_raw['time_utc'])
    train_df = build_features(df_raw[df_raw['time_utc'] < month_start].copy())
    valid_df = build_features(
        df_raw[(df_raw['time_utc'] >= month_start) &
               (df_raw['time_utc'] < next_month_start)].copy()
    )

    LOOKBACK = 24
    TFT_KNOWN = [c for c in
        ['temperature_2m','apparent_temperature','relative_humidity_2m',
         'precipitation','cloud_cover','wind_speed_10m','shortwave_radiation',
         'cdd_65f','hdd_65f']
        if c in train_df.columns]

    def make_tft_df(df):
        out = df.sort_values(['region','time_utc']).copy()
        out['time_idx'] = out.groupby('region').cumcount()
        out['region'] = out['region'].astype(str)
        out['load_mw'] = out['load_mw'].astype(float)
        return out

    tft_train = make_tft_df(train_df)
    tft_valid = make_tft_df(valid_df)
    max_tidx = tft_train.groupby('region')['time_idx'].max()
    tft_valid['time_idx'] = tft_valid.apply(
        lambda r: r['time_idx'] + max_tidx[r['region']] + 1, axis=1
    )
    all_tft = pd.concat([tft_train, tft_valid], ignore_index=True)
    cut = tft_train['time_idx'].max()

    ds_train = TimeSeriesDataSet(
        all_tft[all_tft['time_idx'] <= cut],
        time_idx='time_idx', target='load_mw', group_ids=['region'],
        min_encoder_length=LOOKBACK//2, max_encoder_length=LOOKBACK,
        min_prediction_length=1, max_prediction_length=1,
        static_categoricals=['region'],
        time_varying_known_reals=TFT_KNOWN,
        time_varying_unknown_reals=['load_mw'],
        target_normalizer=GroupNormalizer(groups=['region'], transformation='softplus'),
        add_relative_time_idx=True, add_target_scales=True, add_encoder_length=True,
    )
    ds_valid = TimeSeriesDataSet.from_dataset(ds_train, all_tft, predict=True, stop_randomization=True)
    train_loader = ds_train.to_dataloader(train=True,  batch_size=128, num_workers=2)
    valid_loader = ds_valid.to_dataloader(train=False, batch_size=128, num_workers=2)

    tft_model = TemporalFusionTransformer.from_dataset(
        ds_train, learning_rate=1e-3, hidden_size=64, attention_head_size=4,
        dropout=0.1, hidden_continuous_size=32, loss=TFT_MAPE(),
        log_interval=50, reduce_on_plateau_patience=3,
    )
    print(f'TFT params: {sum(p.numel() for p in tft_model.parameters()):,}')

    trainer = L.Trainer(
        max_epochs=10, accelerator='auto', enable_progress_bar=True,
        gradient_clip_val=0.1, logger=False, enable_checkpointing=False,
    )
    trainer.fit(tft_model, train_dataloaders=train_loader, val_dataloaders=valid_loader)

    raw_preds, _ = tft_model.predict(valid_loader, return_index=True, return_x=False)
    tft_preds  = raw_preds.numpy().flatten()
    tft_actual = valid_df.sort_values(['region','time_utc'])['load_mw'].values
    n    = min(len(tft_preds), len(tft_actual))
    mask = tft_actual[:n] > 0
    tft_mape = np.mean(np.abs(tft_preds[:n][mask] - tft_actual[:n][mask]) / tft_actual[:n][mask]) * 100
    tft_rmse = np.sqrt(np.mean((tft_preds[:n] - tft_actual[:n])**2))
    print(f'\nTFT  MAPE={tft_mape:.2f}%  RMSE={tft_rmse:.1f} MW')

    tft_pred_out = f'{OUTPUT_DIR}/tft_{PREDICT_MONTH}_predictions.csv'
    tft_met_out  = f'{OUTPUT_DIR}/tft_{PREDICT_MONTH}_metrics.csv'
    pd.DataFrame({'tft_pred': tft_preds[:n], 'actual': tft_actual[:n]}).to_csv(tft_pred_out, index=False)
    pd.DataFrame([{'model':'tft','MAPE':tft_mape,'RMSE':tft_rmse}]).to_csv(tft_met_out, index=False)
    print(f'Saved to {tft_pred_out}')

## 4. Compare all results

In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt

metrics_files = sorted(glob.glob(f'{OUTPUT_DIR}/*_{PREDICT_MONTH}_metrics.csv'))
if not metrics_files:
    print('No metrics files found yet. Run some model cells first.')
else:
    rows = []
    for fp in metrics_files:
        mname = os.path.basename(fp).replace(f'_{PREDICT_MONTH}_metrics.csv', '')
        m = pd.read_csv(fp)
        row = {'model': mname}
        row.update(m.iloc[0].to_dict() if len(m) == 1 else m.mean(numeric_only=True).to_dict())
        rows.append(row)

    summary = pd.DataFrame(rows).set_index('model')
    cols = [c for c in ['MAPE','RMSE','MAE'] if c in summary.columns]
    if 'MAPE' in summary.columns:
        summary = summary.sort_values('MAPE')
    print(f'Results for {PREDICT_MONTH}\n')
    display(summary[cols].round(3) if cols else summary.round(3))

    if 'MAPE' in summary.columns:
        fig, ax = plt.subplots(figsize=(10, max(3, len(summary)*0.4)))
        summary['MAPE'].plot(kind='barh', ax=ax)
        ax.set_xlabel('MAPE (%)')
        ax.set_title(f'Model MAPE — {PREDICT_MONTH}')
        ax.axvline(x=5, color='green', linestyle='--', label='5% target')
        ax.legend()
        plt.tight_layout()
        plt.show()

## 5. Prediction plot
Set `PLOT_MODEL` to whichever model you want to inspect.

In [ ]:
PLOT_MODEL  = 'gam'
PLOT_REGION = None  # None = first region found

pred_file = f'{OUTPUT_DIR}/{PLOT_MODEL}_{PREDICT_MONTH}_predictions.csv'
if not os.path.exists(pred_file):
    print(f'No predictions file for {PLOT_MODEL}. Run that model cell first.')
else:
    preds = pd.read_csv(pred_file)
    if 'time_utc' in preds.columns:
        preds['time_utc'] = pd.to_datetime(preds['time_utc'])
    regions = preds['region'].unique() if 'region' in preds.columns else [None]
    region  = PLOT_REGION or regions[0]
    grp     = preds[preds['region'] == region] if region else preds
    x_col   = 'time_utc' if 'time_utc' in grp.columns else grp.index

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(grp[x_col].values, grp['actual'].values,    label='Actual', lw=1)
    ax.plot(grp[x_col].values, grp['predicted'].values, label=PLOT_MODEL.upper(), lw=1, alpha=0.8)
    ax.set_title(f'{PLOT_MODEL.upper()} vs Actual — {region} — {PREDICT_MONTH}')
    ax.set_ylabel('Load (MW)')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 6. Run all models in one shot
Uncomment all lines and run — equivalent to `scripts/train_forecasters.sh`.  
Expected time on T4 GPU: **40–60 min** for all 13 models.

In [ ]:
# run_model('linear')
# run_model('ridge',           '--ridge_alpha 10.0')
# run_model('hinge_regression','--svr_c 1.0 --svr_epsilon 0.1 --svr_max_iter 5000')
# run_model('xgboost')
# run_model('random_forest',   '--rf_n_estimators 300')
# run_model('lightgbm',        '--lgbm_n_estimators 300 --lgbm_learning_rate 0.05 --lgbm_num_leaves 31')
# run_model('lgbm_xgb',        '--lgbm_n_estimators 300 --lgbm_learning_rate 0.05 --lgbm_num_leaves 31 --xgb_n_estimators 300 --xgb_learning_rate 0.05 --ensemble_weight 0.5')
# run_model('gam',             '--gam_n_splines 25 --gam_lam 0.6')
# run_model('prophet',         '--prophet_changepoint_prior_scale 0.05 --prophet_seasonality_prior_scale 10.0')
# run_model('lstm',            '--lookback 24 --epochs 10 --batch_size 128 --lr 0.001 --hidden_dim 64 --num_layers 2 --dropout 0.1')
# run_model('bilstm',          '--lookback 24 --epochs 10 --batch_size 128 --lr 0.001 --hidden_dim 64 --num_layers 2 --dropout 0.1')
# run_model('transformer',     '--lookback 24 --epochs 10 --batch_size 128 --lr 0.001 --d_model 64 --nhead 4 --num_layers 2 --dim_feedforward 128 --dropout 0.1')
# run_model('stcalnet',        '--lookback 24 --epochs 10 --batch_size 128 --lr 0.001 --hidden_dim 64 --num_layers 2 --cnn_channels 64 --dropout 0.1')
print('Uncomment lines above to run everything at once.')